# From Optimization Models to Operational Decision Systems
This notebook is intended to be used alongside the [slide deck](https://docs.google.com/presentation/d/13W83Cs4Jpj6Y-ahmo_zriLagQP2578wSgu62Bqdewf4/edit?usp=sharing) for the AMPL [workshop](https://meetings.informs.org/wordpress/analytics/exhibitor-workshops/#ampl) presented at [INFORMS Analytics+ Conference 2026](https://meetings.informs.org/wordpress/analytics/).

## Code
This section contains the code needed to run the rest of the notebook. Don't worry too much if it's not completely clear what every function does. It'll be more apparent at the end of the workshop.

**NOTE:** this notebook depends on specific files being present in the same folder as the notebook. In case of using google colab, please make sure to load those.

In [ ]:
import pandas as pd
from collections import defaultdict
import numpy as np
import time 
import matplotlib.pyplot as plt

In [ ]:
# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook

# ampl = ampl_notebook(
#     modules=["highs"],  # modules to install
#     license_uuid="default",  # license to use
# )  # instantiate AMPL object and register magics

ampl = AMPL()

In [ ]:
def load_data_to_model(ampl, scenario="S1", model_name="base_model.mod", scale=False):
    """
    General function to load input data from Excel files and populate the AMPL model.
    Decides which components to load based on the model_name.
    """
    scenario_file = f"data/data_{scenario}.xlsx"
    technical_data_file = "data/technical_data.xlsx"

    dct_data = pd.read_excel(scenario_file, sheet_name=None)
    dct_data = {k: v.round(2) for k, v in dct_data.items()}
    
    dct_technical = pd.read_excel(technical_data_file, sheet_name=None)
    dct_technical = {k: v.round(4) for k, v in dct_technical.items()}

    # region: Base Processing (Always loaded)
    ramps = dct_technical["ramps"].set_index("thermal_gen")
    efficiency = dct_technical["efficiency"].set_index("hydro_gen")
    reservoir_cap = dct_technical["reservoir_cap"].set_index("hydro_gen")

    market_price = dct_data["market_price"].set_index("period")
    marginal_cost = dct_data["marginal_cost"].set_index(["thermal_gen", "period"])
    inflows = dct_data["inflows"].set_index(["hydro_gen", "period"])
    if scale:
        inflows.loc[:,'inflows'] = inflows['inflows'] / 1000  # Scale to match delta_t
    renewable_resource = dct_data["renewable_resource"].set_index(["renewable_gen", "period"])
    max_generation = dct_data["max_generation"].set_index(["con_gen", "period"])

    ROR_HYDRO_GENERATORS = [gen for gen in dct_technical["efficiency"]["hydro_gen"].unique() if "ror" in gen]
    SOLAR_GENERATORS = [gen for gen in dct_data["renewable_resource"]["renewable_gen"].unique() if "solar" in gen]
    WIND_GENERATORS = [gen for gen in dct_data["renewable_resource"]["renewable_gen"].unique() if "wind" in gen]
    # endregion

    # region: Model-Specific logic
    exclude_eff = []
    
    # PWL Efficiency
    if "pwl" in model_name:
        pwl_config = dct_technical["pwl_config"].set_index("hydro_gen")
        pwl_breakpoints = dct_technical["pwl_breakpoints"].set_index(["hydro_gen", "p"])
        pwl_slopes = dct_technical["pwl_slopes"].set_index(["hydro_gen", "p"])
        if scale:
            pwl_config.loc[:,'intercept'] = pwl_config['intercept'] * 1000
            pwl_slopes.loc[:,'slopes'] = pwl_slopes['slopes'] * 1000
        PWL_GENERATORS = pwl_config.index.tolist()
        exclude_eff.extend(PWL_GENERATORS)
        
        ampl.set["RESERVOIR_PWL_EFF_GENERATORS"] = PWL_GENERATORS
        ampl.set_data(pwl_config)
        ampl.set_data(pwl_breakpoints)
        ampl.set_data(pwl_slopes)

    # Quadratic/Smooth Efficiency
    if "nlp" in model_name:
        quad_coeff = dct_technical["quad_coeff"].set_index(["hydro_gen", "p"])
        QUAD_GENERATORS = dct_technical["quad_coeff"]["hydro_gen"].unique().tolist()
        exclude_eff.extend(QUAD_GENERATORS)
        
        ampl.set["RESERVOIR_QUAD_EFF_GENERATORS"] = QUAD_GENERATORS
        if scale:
            quad_coeff.loc[:,'quad_coeff'] = quad_coeff['quad_coeff'] * 1000
        ampl.set_data(quad_coeff)

    # BESS
    if "bess" in model_name:
        bess_data = dct_technical["bess"].set_index("bess_id")
        if "rules" not in model_name:
            bess_data.drop(columns=["min_charging_periods", "min_charging_power"], inplace=True)         
        ampl.set_data(bess_data, set_name="BESS")

    # Unit Commitment
    if "uc" in model_name:
        thermal_uc = dct_technical["thermal_uc"].set_index("thermal_gen")
        ampl.set_data(thermal_uc)

    # Final Efficiency filtering
    efficiency_standard = efficiency[~efficiency.index.isin(list(set(exclude_eff)))]
    if scale:
        efficiency_standard.loc[:,'efficiency'] = efficiency_standard['efficiency'] * 1000
    # endregion

    # region: Standard Loading to AMPL
    ampl.set["ROR_HYDRO_GENERATORS"] = ROR_HYDRO_GENERATORS
    ampl.set["SOLAR_GENERATORS"] = SOLAR_GENERATORS
    ampl.set["WIND_GENERATORS"] = WIND_GENERATORS

    ampl.set_data(ramps, set_name="THERMAL_GENERATORS")
    ampl.set_data(reservoir_cap, set_name="RESERVOIR_HYDRO_GENERATORS")
    ampl.set_data(market_price, set_name="PERIODS")

    # Topology
    upstream_set_collection = ampl.set["UPSTREAM_GENERATORS"]
    upstream_set_collection["reservoir1"] = ["reservoir3"]

    # Parameters        
    ampl.set_data(marginal_cost)
    ampl.set_data(inflows)
    ampl.set_data(renewable_resource)
    ampl.set_data(max_generation)
    ampl.set_data(efficiency_standard)

    if scale:
        ampl.param['period_duration'] = 3600  # 1 hour in seconds
        ampl.param['delta_t'] = ampl.param['period_duration'].value() / 1000
    # endregion

def solve_model(ampl, scenario="S1", model_name="base_model.mod", export_model=False, verbose=False, scale=False, **kwargs):
    """
    General solve function for any model variant.
    """
    ampl.reset()
    ampl.read(model_name)

    load_data_to_model(ampl, scenario=scenario, model_name=model_name, scale=scale)

    ampl.eval("reset options;")
    for k, v in kwargs.items():
        ampl.set_option(k, v)
    
    start = time.perf_counter()
    ampl.solve(verbose=verbose)
    end = time.perf_counter()
    print(f"Solve time: {int((end - start)/60)} minutes and {int((end - start)%60)} seconds")

    status = ampl.solve_result
    if "solved" in status or "limit" in status:
        try:
            # Check which objective is defined
            obj_name = "TotalProfit"
            
            objective = ampl.get_objective(obj_name).value()
            print(f"Status: {status}. Total Profit ({scenario} - {model_name}): {round(objective/1e6, 2)} MUSD.\n")
        except Exception as e:
            print(f"Status: {status}. Model solved but error retrieving profit: {e}")
    else:
        print(f"Optimization failed or returned status: {status}")

    if export_model:
        ampl.snapshot(f"model_snapshot_{scenario}.run")


In [ ]:
def plot_bess_results(ampl, daystart=115, dayend=121):
        df = ampl.get_data("SoC, Charge, {g in BESS, t in PERIODS} -1*Discharge[g,t]").to_pandas().reset_index()
        df.columns = ['bess', 'period', 'soc', 'charge', 'discharge']

        start_hour = (daystart - 1) * 24 + 1
        end_hour = dayend * 24
        df_filtered = df[(df['period'] >= start_hour) & (df['period'] <= end_hour)]

        # 1. Create figure and axes
        fig, ax = plt.subplots(figsize=(3.5,2.5))

        # 2. Plot the lines with specific colors matching the target image
        # Blue for charge, Green for discharge
        ax.plot(df_filtered['period'], df_filtered['charge'], 
                color='#1f77b4', label='Charge Power', linewidth=1.5)
        ax.plot(df_filtered['period'], df_filtered['discharge'], 
                color='#00a950', label='Discharge Power', linewidth=1.5)

        # 3. Set title and axis labels
        ax.set_ylabel('Power (KW)', fontsize=9, fontweight='bold')
        ax.set_xlabel('Time (Days)', fontsize=9, fontweight='bold')

        # 4. Configure grid lines (horizontal only, rendered behind the data)
        ax.yaxis.grid(True, color='#e0e0e0', linestyle='-', linewidth=1)
        ax.xaxis.grid(False)
        ax.set_axisbelow(True) # Ensures grid lines stay behind the plotted lines

        # 5. Configure X-axis ticks to show "Day 1", "Day 2", etc.
        tick_positions = range(start_hour, end_hour + 1, 24)
        tick_labels = [f"Day {i+1}" for i in range(len(tick_positions))]

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels)
        ax.tick_params(axis='both', labelsize=8)
        ax.tick_params(axis='x', rotation=30)

        # 6. Style the spines (the bounding box of the plot)
        for spine in ax.spines.values():
                spine.set_color('gray')
                spine.set_linewidth(0.8)

        # 7. Add and style the legend
        ax.legend(loc='upper right', frameon=True, edgecolor='darkgray', fontsize=8)

        # Adjust layout to prevent clipping
        plt.tight_layout()


## Problem Statement

Throughout the workshop, you'll be working with an example from the energy industry.

The case is designed to be as close to reality as possible. All the data is based on a power generation portfolio from a real utility. Furthermore, the scenarios used as inputs correspond to publicly available historical data from the market this company operates in and its plants related information. 

In liberalized electricity markets, generation companies face a complex daily challenge: how to operate a diverse fleet of power plants to maximize profit without violating physical or environmental constraints. It is not uncommon for large market participants to own a portfolio of several power generation technologies (e.g. big hydro with reservoirs, small run-of-river hydro, solar, wind, thermal plants, batteries, among others) and correctly managing them is not a trivial task. This problem is refered to as **Power Generation Portfolio Optimization**. The objective is to maximize Producer Surplus, defined mathematically as the difference between the revenue gained from selling energy at market prices and the variable costs of production (fuel and operations).

The challenge lies in managing the several assets with their own competing characteristics. Firstly, **thermal units** (coal, gas, nuclear) provide stable baseload power but come with **high marginal costs and physical inertia**. They consume fuel for every MWh produced and cannot change output instantly. The model should account for ramp constraints, forcing the optimizer to "plan ahead" for peak price hours rather than reacting instantly. On the other hand, **hydropower** is the most complex asset to manage because **it can affect both time and space dimensions**. Time is coupled through reservoirs where one can store water (energy) today to use in the future. Besides that, rivers often have multiple generators in a chain. In this sense, water released from an "Upstream" dam flows down to become inflow for a "Downstream" one. This creates a dependency where the generation of one unit affects the resource availability of another. Finally, **renewables** like solar and wind are **zero-marginal-cost resources** whose energy, ideally, must be produced whenever it is available.

Throughout this notebook reservoir and dam are used indistinctively.

A complete mathematical description can be found [here](https://colab.ampl.com/notebooks/power-generation-portfolio-optimization.html#model)

## Step 1. Moving Beyong Hardcoded models

Often, you'll start with a quick prototype filled with data and lacking the proper structure. 

In [ ]:
%%writefile wrong_model.mod
# 🚩 No sections. Everything is mixed together.
var g_t {1..24} >= 0; # 🚩 Terrible, cryptic naming
param c {1..24};
var g_s {1..24} >= 0;
param r_s {1..24};
# 🚩 Hardcoded scalars mixed into declarations
param eff = 0.85;
var v {0..24} >= 0; # 🚩 Hardcoded time indices (1..24)
var q {1..24} >= 0;
var sp {1..24} >= 0;
param p {1..24};
maximize z: # 🚩 Meaningless objective name
sum{t in 1..24} (p[t] * (g_t[t] + g_s[t] + (eff*q[t])))
- sum{t in 1..24} (c[t]*g_t[t])
- sum{t in 1..24} (250*sp[t]); # 🚩 Hardcoded penalty inside objective
subject to c1 {t in 1..24}: g_t[t] <= 100; # 🚩 Unhelpful constraint names
subject to c2 {t in 1..24}: g_s[t] <= r_s[t];
subject to c3 {t in 1..24}:
# 🚩 Clunky index math (t-1) and hardcoded unit conversions
v[t] = v[t-1] + 0.0036 * (15 - q[t] - sp[t]);
# 🚩 Initial/Final conditions hardcoded as brittle constraints
subject to c4: v[0] = 0.7 * 500;
subject to c5: v[24] = 350;
# 🚩 Clunky subset bounding for ramps
subject to c6 {t in 2..24}: g_t[t] - g_t[t-1] <= 40;


Although the syntax is correct, and the code would not produce any errors, there are many things that can be improved from this formulation.

In [ ]:
%%writefile correct_model.mod
# ✅ Clear sections separating structure, data, and logic.
### SETS
# ✅ Ordered sets replace brittle, hardcoded indices like {1..24}.
set PERIODS ordered;
### PARAMETERS
# ✅ Verbose, self-documenting naming conventions.
param thermal_cost {PERIODS}; param solar_resource {PERIODS}; param inflows {PERIODS}; param market_price {PERIODS};
# ✅ Scalars parameterized with safe cascading defaults. No hardcoded values.
param hydro_efficiency > 0 default 0.85; param delta_t default 0.0036; param penalty_spillage >= 0 default 250;
param max_thermal_cap >= 0 default 100; param ramp_limit >= 0 default 40;
### OBJECTIVE
# ✅ Meaningful objective name and parameterized penalties.
maximize Total_Profit: sum{t in PERIODS} (market_price[t] * (Thermal_Gen[t] + Solar_Gen[t] + (hydro_efficiency * TurbiningFlow[t])))
- sum{t in PERIODS} (thermal_cost[t] * Thermal_Gen[t]) - sum{t in PERIODS} (penalty_spillage * SpillageFlow[t]);
### CONSTRAINTS
# ✅ Special ordered set functions (first, prev) eliminate clunky t-1 math.
subject to Water_Balance {t in PERIODS}: ReservoirStorage[t] = (if t = first(PERIODS) then initial_storage else ReservoirStorage[prev(t)])
+ delta_t * (inflows[t] - TurbiningFlow[t] - SpillageFlow[t]);
# ✅ Final condition uses last() function, robust to horizon changes.
subject to Final_Storage_Target: ReservoirStorage[last(PERIODS)] = target_storage;
# ✅ Clean subset bounding using ord() to safely manage ramps.
subject to Thermal_Ramp_Up {t in PERIODS: ord(t) > 1}: Thermal_Gen[t] - Thermal_Gen[prev(t)] <= ramp_limit;


Following the best practices [guide](https://dev.ampl.com/ampl/best-practices/amplpy-best-practices.html) can get you working on scalable models from day 1.

For our energy case, the base model would look like this:

In [ ]:
%%writefile base_model.mod
# Model Name: Energy Portfolio Optimization (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;


### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)

# Hydro Params
param efficiency {HYDRO_GENERATORS} > 0;               # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)

# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s


### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS
subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = efficiency[g] * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = efficiency[g] * SpillageFlow[g, t];

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to ThermalRampUp {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= ramp_up_rate[g];

subject to ThermalRampDown {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= ramp_down_rate[g];

subject to GenerationMaxLimit {g in CONVENTIONAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

subject to GenerationMinLimit {g in GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];



Notice the correct `set` usage and adequate structure. The use of `default` values and AMPL syntax make this version truly robust, flexible and scalable.

The **lesson** here is to always build thinking in production and scalability. 

Always be aware of the red flags that make models hard to scale and difficult to work with. Don't waste time with flawed prototypes, take advantage of AMPL's clear syntax to produce top notch models

In [ ]:
solvers = ["highs"]  # ["gurobi", "xpress", "cplex", "knitro", "highs"]
scenarios = [f"S{i}" for i in range(1, 6)]  # Choose which scenarios to run
for sol in solvers:
    print(f"Solving with {sol}...\n")
    start = time.perf_counter()

    for s in scenarios:
        print(f"Solving scenario {s}...")
        options = {
            "solver": sol, 
            f"{sol}_options": "outlev=1",
        }
        solve_model(ampl, scenario=s, model_name='base_model.mod', verbose=False, **options)
    end = time.perf_counter()
    print(f"Elapsed time: {int((end - start)/60)} minutes and {int((end - start)%60)} seconds\n")


## Step 2. Modeling Real World Complexities

When building decision science or optimization solutions, requirements are never static. You'll have to adapt quickly to new complexities that will certainly arise. In this section we'll explore three adjustments to our generation portfolio model:
1. Modeling the (non-linear) relationship between reservoir volume and conversion factor or efficiency.
2. Including a battery energy storage system (BESS) in the portfolio.
3. Adding specific UC-like contraints to thermal units such as minimum online/offline time

### 2.1 Reservoir level vs Conversion Factor

The efficiency of the hydraulic plant is measured in MW/m3/s and indicates how much power can be produced per unit of water inflow.

In the case of reservoirs, the height or volume level of the dam affect this value directly. However, it's not a trivial task to establish this relationship since it depends on the dam structure, shape and type.

In practice, several real operation pairs of dam level vs measured efficiency are taken and used to estimate a relationship.

Notice that this value is used in the model to get the generation of the plant, by multiplying it by the turbined flow. If efficiency becomes a function of the storage, the model becomes non-linear, in spite of the method chosen to model the relationship.

We'll first try to model it using piecewise linear approximations (PWL)

![PWL approximation](img/pwl.jpg)

In [ ]:
%%writefile base_pwl.mod
# Model Name: Energy Portfolio Optimization (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set RESERVOIR_PWL_EFF_GENERATORS within RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;


### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)

# Hydro Params
param efficiency {HYDRO_GENERATORS diff RESERVOIR_PWL_EFF_GENERATORS} > 0;               # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)
# For PWL efficiency
param npiece {RESERVOIR_PWL_EFF_GENERATORS} integer >= 1 default 5;  # Number of pieces for PWL approximation
param breakpoints {g in RESERVOIR_PWL_EFF_GENERATORS, p in 1..npiece[g]}; # Breakpoints for reservoir storage (Hm3/1000)
param slopes {g in RESERVOIR_PWL_EFF_GENERATORS, p in 1..npiece[g]+1};  # Slopes for each segment of the PWL efficiency (MW/m3/s per segment)
param intercept {RESERVOIR_PWL_EFF_GENERATORS}; # Intercept for the PWL efficiency function (MW/m3/s)


# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s

# Efficiency as a function of Reservoir Storage (PWL)
var ScaledReservoirStorage {g in RESERVOIR_PWL_EFF_GENERATORS, t in PERIODS} = ReservoirStorage[g,t]/1000; # Scale to match breakpoints
var Efficiency_PWL {g in RESERVOIR_PWL_EFF_GENERATORS, t in PERIODS} = 
    (<<{p in 1..npiece[g]} breakpoints[g,p]; {p in 1..npiece[g]+1} slopes[g,p]>> 
        ScaledReservoirStorage[g,t]) + intercept[g];

### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS
subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = (if g in RESERVOIR_PWL_EFF_GENERATORS 
                        then Efficiency_PWL[g, t] 
                        else efficiency[g]) * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = (if g in RESERVOIR_PWL_EFF_GENERATORS 
                      then Efficiency_PWL[g, t] 
                      else efficiency[g]) * SpillageFlow[g, t];

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to ThermalRampUp {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= ramp_up_rate[g];

subject to ThermalRampDown {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= ramp_down_rate[g];

subject to GenerationMaxLimit {g in CONVENTIONAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

subject to GenerationMinLimit {g in GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];


As you can see [here](https://www.diffchecker.com/es/INQNMxVc/), modeling PWL relationships is a trivial task in AMPL. You just need the breakpoints and slopes for each segment. Moreover, the model itself didn't change that much.

However, this is not the only solution to the problem. Using the same data points, a polynomic regression can be applied to estimate a quadratic relationship between the variables.

![quadratic approximation](img/quad.jpg)

In [ ]:
%%writefile base_smooth_nlp.mod
# Model Name: Energy Portfolio Optimization (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set RESERVOIR_QUAD_EFF_GENERATORS within RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;


### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)

# Hydro Params
param efficiency {HYDRO_GENERATORS diff RESERVOIR_QUAD_EFF_GENERATORS} > 0;               # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)
# For QUAD efficiency
param quad_coeff {g in RESERVOIR_QUAD_EFF_GENERATORS, p in 1..3};  # Quadratic coefficient for efficiency function

# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s

# Efficiency as a function of Reservoir Storage (QUAD)
var ScaledReservoirStorage {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = ReservoirStorage[g,t]/1000; # Scale to match function
var Efficiency_QUAD {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = 
    quad_coeff[g,1] * (ScaledReservoirStorage[g,t])^2 + quad_coeff[g,2] * ScaledReservoirStorage[g,t] + quad_coeff[g,3];

### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS
subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                        then Efficiency_QUAD[g, t] 
                        else efficiency[g]) * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                      then Efficiency_QUAD[g, t] 
                      else efficiency[g]) * SpillageFlow[g, t];

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to ThermalRampUp {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= ramp_up_rate[g];

subject to ThermalRampDown {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= ramp_down_rate[g];

subject to GenerationMaxLimit {g in CONVENTIONAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

subject to GenerationMinLimit {g in GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];


Once again, model changes are [minimal](https://www.diffchecker.com/es/wekIrlhz/).

Performance of both approaches will be compared in the following sections.

There are additional ways to model the relationship that are not covered in the notebook. One of those is using a succesive linear programming (SLP) technique, in which the model is solved iteratively setting a constant efficiency value for each period based on the solution of the previous iteration. This alternative would practically not change the base model but would require several solves of it. 

### 2.2 Including BESS in the portfolio

As storage systems, BESS and reservoirs share some functionalities. Nevertheless, dams are used for medium to long term decision while battery systems docus on short term revenue opportunities, doing price arbitrage where possible.

On the other hand, BESS are often used to participate in ancillary services market given their rapid response charactersitics. Modeling these markets is out of the scope of the current workshop. However, in order to do so binary variables are typically added to represent the battery state at each period (charging, discharging).

In [ ]:
%%writefile base_nlp_bess.mod
# Model Name: Energy Portfolio Optimization (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set RESERVOIR_QUAD_EFF_GENERATORS within RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;
set BESS default {};
set PORTFOLIO = GENERATORS union BESS;


### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)

# Hydro Params
param efficiency {HYDRO_GENERATORS diff RESERVOIR_QUAD_EFF_GENERATORS} > 0;               # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)
# For QUAD efficiency
param quad_coeff {g in RESERVOIR_QUAD_EFF_GENERATORS, p in 1..3};  # Quadratic coefficient for efficiency function

# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# BESS params
param max_bess_power {BESS} >= 0;  # MW
param bess_capacity {BESS} >= 0;  # MWh
param bess_efficiency {BESS} > 0, <= 1 default 0.85;  # Round trip efficiency
param initial_soc {b in BESS} >= 0 default 0.5*bess_capacity[b]; # Initial state of charge MWh


# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s

# Efficiency as a function of Reservoir Storage (QUAD)
var ScaledReservoirStorage {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = ReservoirStorage[g,t]/1000; # Scale to match function
var Efficiency_QUAD {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = 
    quad_coeff[g,1] * (ScaledReservoirStorage[g,t])^2 + quad_coeff[g,2] * ScaledReservoirStorage[g,t] + quad_coeff[g,3];

# BESS Variables
var Charge {BESS, PERIODS} >= 0;     # MW
var Discharge {BESS, PERIODS} >= 0;  # MW
var SoC {g in BESS, t in PERIODS} >= 0 <= bess_capacity[g];        # State of Charge (MWh)
var IsCharging {BESS, PERIODS} binary;
var IsDischarging {BESS, PERIODS} binary;

### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    + sum {b in BESS, t in PERIODS} (market_price[t] * Discharge[b, t])
    - sum {b in BESS, t in PERIODS} (market_price[t] * Charge[b, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS
subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                        then Efficiency_QUAD[g, t] 
                        else efficiency[g]) * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                      then Efficiency_QUAD[g, t] 
                      else efficiency[g]) * SpillageFlow[g, t];

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to ThermalRampUp {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= ramp_up_rate[g];

subject to ThermalRampDown {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= ramp_down_rate[g];

subject to GenerationMaxLimit {g in CONVENTIONAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

subject to GenerationMinLimit {g in GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];

subject to ChargeLimit {g in BESS, t in PERIODS}:
    Charge[g, t] <= max_bess_power[g] * IsCharging[g, t];

subject to DischargeLimit {g in BESS, t in PERIODS}:
    Discharge[g, t] <= max_bess_power[g] * IsDischarging[g, t];

subject to ChargingStateExclusion {g in BESS, t in PERIODS}:
    IsCharging[g, t] + IsDischarging[g, t] <= 1;

subject to SoC_Balance {g in BESS, t in PERIODS}:
    SoC[g, t] = (if t = first(PERIODS) then initial_soc[g] else SoC[g, prev(t)])
                + (Charge[g, t] * (bess_efficiency[g]^0.5)) 
                - (Discharge[g, t] / (bess_efficiency[g]^0.5));

Only a [few](https://www.diffchecker.com/es/UO8nxthK/) changes to the previous version of the model were necessary.

In [ ]:
solvers = ["gurobi"]
scenarios = [f"S{i}" for i in range(1, 6)]  # Choose which scenarios to run
for sol in solvers:
    print(f"Solving with {sol}...\n")
    start = time.perf_counter()

    for s in scenarios:
        print(f"Solving scenario {s}...")
        options = {
            "solver": sol,
            f"{sol}_options": "outlev=1 mipgap=0.01 timelimit=120",
        }
        solve_model(ampl, scenario=s, model_name='base_nlp_bess.mod', verbose=False, **options)
    end = time.perf_counter()
    print(f"Elapsed time: {int((end - start)/60)} minutes and {int((end - start)%60)} seconds\n")


In [ ]:
plot_bess_results(ampl)

Results show a frequent change of state for the BESS. This can damage battery life and so, new operational rules are required so that once the battery start charging it stays in that state for a minimum number of periods.

In [ ]:
%%writefile base_nlp_bess_rules.mod
# Model Name: Energy Portfolio Optimization (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set RESERVOIR_QUAD_EFF_GENERATORS within RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;
set BESS default {};
set PORTFOLIO = GENERATORS union BESS;


### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)

# Hydro Params
param efficiency {HYDRO_GENERATORS diff RESERVOIR_QUAD_EFF_GENERATORS} > 0;               # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)
# For QUAD efficiency
param quad_coeff {g in RESERVOIR_QUAD_EFF_GENERATORS, p in 1..3};  # Quadratic coefficient for efficiency function

# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# BESS params
param max_bess_power {BESS} >= 0;  # MW
param bess_capacity {BESS} >= 0;  # MWh
param bess_efficiency {BESS} > 0, <= 1 default 0.85;  # Round trip efficiency
param initial_soc {b in BESS} >= 0 default 0.5*bess_capacity[b]; # Initial state of charge MWh
param min_charging_periods {BESS} >= 0 default 3; # Minimum number of consecutive periods that the BESS must charge when it starts
param min_charging_power {BESS} >= 0 default 10; # Minimum charging power when the BESS is charging

# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s

# Efficiency as a function of Reservoir Storage (QUAD)
var ScaledReservoirStorage {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = ReservoirStorage[g,t]/1000; # Scale to match function
var Efficiency_QUAD {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = 
    quad_coeff[g,1] * (ScaledReservoirStorage[g,t])^2 + quad_coeff[g,2] * ScaledReservoirStorage[g,t] + quad_coeff[g,3];

# BESS Variables
var Charge {BESS, PERIODS} >= 0;     # MW
var Discharge {BESS, PERIODS} >= 0;  # MW
var SoC {g in BESS, t in PERIODS} >= 0 <= bess_capacity[g];        # State of Charge (MWh)
var IsCharging {BESS, PERIODS} binary;
var IsDischarging {BESS, PERIODS} binary;

### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    + sum {b in BESS, t in PERIODS} (market_price[t] * Discharge[b, t])
    - sum {b in BESS, t in PERIODS} (market_price[t] * Charge[b, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS
subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                        then Efficiency_QUAD[g, t] 
                        else efficiency[g]) * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                      then Efficiency_QUAD[g, t] 
                      else efficiency[g]) * SpillageFlow[g, t];

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to ThermalRampUp {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= ramp_up_rate[g];

subject to ThermalRampDown {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= ramp_down_rate[g];

subject to GenerationMaxLimit {g in CONVENTIONAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

subject to GenerationMinLimit {g in GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];

subject to ChargeMaxLimit {g in BESS, t in PERIODS}:
    Charge[g, t] <= max_bess_power[g] * IsCharging[g, t];

subject to ChargeMinLimit {g in BESS, t in PERIODS}:
    Charge[g, t] >= min_charging_power[g] * IsCharging[g, t];

subject to DischargeLimit {g in BESS, t in PERIODS}:
    Discharge[g, t] <= max_bess_power[g] * IsDischarging[g, t];

subject to ChargingStateExclusion {g in BESS, t in PERIODS}:
    IsCharging[g, t] + IsDischarging[g, t] <= 1;
    
subject to SoC_Balance {g in BESS, t in PERIODS}:
    SoC[g, t] = (if t = first(PERIODS) then initial_soc[g] else SoC[g, prev(t)])
                + (Charge[g, t] * (bess_efficiency[g]^0.5)) 
                - (Discharge[g, t] / (bess_efficiency[g]^0.5));

subject to MinChargingPeriods {g in BESS, t in PERIODS: ord(t) > 1}:
    (IsCharging[g,t] == 1 and IsCharging[g, prev(t)] == 0) ==> forall {tp in t..min(t+min_charging_periods[g]-1, last(PERIODS))} IsCharging[g, tp] == 1;

Thanks to AMPL's MP reformulations and logic syntax, the adjustment only required to define the new parameter and add one constraint to enforce the rule, providing better and more realistic results.

#### Under the hood: how `MinChargingPeriods` is reformulated

The rule is a logical implication: *if the BESS starts charging, it must keep charging for `min_charging_periods`*; most solvers cannot accept that directly, so AMPL's MP library reformulates it before the solver sees it, using the facilities available for the solver currently in use.

MP can export this reformulation graph with the `writegraph` option, which lets us look at it.

To keep the graph readable we use a **simplified instance** of `base_nlp_bess_rules.mod`: one BESS, 8 hours, no generators, and a minimum charging run of 3 periods.

In [ ]:
# Simplified instance: 1 BESS, 8 periods, no generators
prices = pd.DataFrame({"period": range(1, 9),
                       "market_price": [30, 20, 15, 18, 45, 80, 60, 25]}).set_index("period")
bess = pd.read_excel("data/technical_data.xlsx", sheet_name="bess").set_index("bess_id")
bess["min_charging_periods"] = 3

def export_reformulation_graph(solver, filename):
    """Load the simplified instance and write MP's reformulation graph to `filename`."""
    ampl.reset()
    ampl.read("base_nlp_bess_rules.mod")
    ampl.set_data(prices, set_name="PERIODS")
    ampl.set_data(bess, set_name="BESS")
    ampl.set_option(f"{solver}_auxfiles", "rc")  # keep variable and constraint names
    # lim:time=0 -> we only want the export, not a solution (sol:chk:mode=0 skips the solution check)
    ampl.solve(solver=solver, **{f"{solver}_options": f"writegraph={filename} lim:time=0 sol:chk:mode=0"})

export_reformulation_graph("gurobi", "bess_rules_gurobi.jsonl")
export_reformulation_graph("highs", "bess_rules_highs.jsonl")

In [ ]:
from reformulation_tree import show
# Remind us of the constraint
print("(ΙsCharging[g,t] == 1 and IsCharging[g, prev(t)] == 0) ==> forall {tp in t..min(t+min_charging_periods[g]-1, last(PERIODS))} IsCharging[g, tp] == 1;)")

show("bess_rules_gurobi.jsonl", search="MinChargingPeriods['bess1',3]")



How to read the tree, top to bottom:

* The first line is the constraint as AMPL passes it to MP. The implication `A ==> B` is already the disjunction `not A or B`, where:
    * **A** = ```(ΙsCharging[g,t] == 1 and IsCharging[g, prev(t)] == 0) ``` 
    * **B** = ```forall {tp in t..min(t+min_charging_periods[g]-1, last(PERIODS))} IsCharging[g, tp] == 1;)```
* MP gives every sub-expression its own **`flatv` variable**: a `Not` for the first condition, an `And` for the three periods of the window, and an `Or` over everything, fixed to 1 because the constraint must hold.
* **Greyed-out** items are intermediate steps that were merged or replaced. They are not in the final solver model.
* The items that are not greyed out are what the solver receives. Gurobi has native `And` / `Or` constraints, so they are passed as they are.

A solver without native logical constraints needs a different reformulation. Same instance with HiGHS:

In [ ]:
show("bess_rules_highs.jsonl", search="MinChargingPeriods['bess1',3]")

With HiGHS the `And` and `Or` are expanded into **linear inequalities** (the `linle` / `linge` leaves). The model in AMPL does not change; MP chooses the reformulation that suits the solver.

#### Execution of BESS model

In [ ]:
solvers = ["gurobi"]
scenarios = [f"S{i}" for i in range(1, 6)]  # Choose which scenarios to run
for sol in solvers:
    print(f"Solving with {sol}...\n")
    start = time.perf_counter()

    for s in scenarios:
        print(f"Solving scenario {s}...")
        options = {
            "solver": sol,
            f"{sol}_options": "outlev=1 mipgap=0.01 timelimit=120",
        }
        solve_model(ampl, scenario=s, model_name='base_nlp_bess_rules.mod', verbose=False, **options)
    end = time.perf_counter()
    print(f"Elapsed time: {int((end - start)/60)} minutes and {int((end - start)%60)} seconds\n")


In [ ]:
plot_bess_results(ampl)

### 2.3 Unit Commitment-like constraints for Themal Units

Thermal generation units usually must comply with certain rules of operation. One of the most common ones is having a minimum online or offline time. This can be accomplished in a similar manner than before or with more traditional modeling math.

In [ ]:
%%writefile uc_bess_nlp.mod

# Model Name: Energy Portfolio Optimization with Unit Commitment (pricetaker)

### SETS
set PERIODS ordered;  # hours
set THERMAL_GENERATORS default {};
set RESERVOIR_HYDRO_GENERATORS default {};
set RESERVOIR_QUAD_EFF_GENERATORS within RESERVOIR_HYDRO_GENERATORS default {};
set CONVENTIONAL_GENERATORS = THERMAL_GENERATORS union RESERVOIR_HYDRO_GENERATORS;
set ROR_HYDRO_GENERATORS default {};
set HYDRO_GENERATORS = RESERVOIR_HYDRO_GENERATORS union ROR_HYDRO_GENERATORS;
set UPSTREAM_GENERATORS {HYDRO_GENERATORS} within HYDRO_GENERATORS default {};
set SOLAR_GENERATORS default {};
set WIND_GENERATORS default {};
set RENEWABLE_GENERATORS = SOLAR_GENERATORS union WIND_GENERATORS;
set GENERATORS = THERMAL_GENERATORS union HYDRO_GENERATORS union RENEWABLE_GENERATORS;
set BESS default {};
set PORTFOLIO = GENERATORS union BESS;

### PARAMETERS
# Forecasted Params
param market_price {PERIODS} >= 0;                # Forecasted market energy price ($/MWh)
param marginal_cost {THERMAL_GENERATORS, PERIODS} >= 0;    # Marginal fuel cost ($/MWh)
param inflows {HYDRO_GENERATORS, PERIODS} >= 0 default 0;  # Water contributions/inflows (m3/s)
param renewable_resource {RENEWABLE_GENERATORS, PERIODS} >= 0; # Renewable resource availability (MW)

# Thermal Units Params (Including New Unit Commitment Params)
param ramp_up_rate {THERMAL_GENERATORS} >= 0;     # Ramp up limit (MW/hour)
param ramp_down_rate {THERMAL_GENERATORS} >= 0;   # Ramp down limit (MW/hour)
param min_up_time {THERMAL_GENERATORS} >= 0 default 1;      # Minimum consecutive periods online
param min_down_time {THERMAL_GENERATORS} >= 0 default 1;    # Minimum consecutive periods offline
param initial_online {THERMAL_GENERATORS} binary default 0; # Initial status before period 1

# Hydro Params
param efficiency {HYDRO_GENERATORS diff RESERVOIR_QUAD_EFF_GENERATORS} > 0;                # Conversion Factor (MW/m3/s)
param reservoir_max_capacity {RESERVOIR_HYDRO_GENERATORS} >= 0; # Reservoir max capacity (Hm3)
param initial_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default 0.7*reservoir_max_capacity[r];         # Initial reservoir energy (Hm3)
param final_storage {r in RESERVOIR_HYDRO_GENERATORS} >= 0 default initial_storage[r];  # Required energy at end of horizon (Hm3)
# For QUAD efficiency
param quad_coeff {g in RESERVOIR_QUAD_EFF_GENERATORS, p in 1..3};  # Quadratic coefficient for efficiency function

# Operational Params/Limits
param max_generation {CONVENTIONAL_GENERATORS, PERIODS} >= 0;  # Maximum generation (MW)
param min_generation {GENERATORS, PERIODS} >= 0 default 0;  # Minimum generation (MW)

# BESS params
param max_bess_power {BESS} >= 0;  # MW
param bess_capacity {BESS} >= 0;  # MWh
param bess_efficiency {BESS} > 0, <= 1 default 0.85;  # Round trip efficiency
param initial_soc {b in BESS} >= 0 default 0.5*bess_capacity[b]; # Initial state of charge MWh
param min_charging_periods {BESS} >= 0 default 3; # Minimum number of consecutive periods that the BESS must charge when it starts
param min_charging_power {BESS} >= 0 default 10; # Minimum charging power when the BESS is charging

# Penalty Params
param penalty_spillage {HYDRO_GENERATORS} >= 0 default 2.5* max{t in PERIODS}(market_price[t]); # Penalty for water spillage ($/MWh)

# Computed/Scalar Params
param period_duration >=0 default 3600; # Duration of each period in seconds (default: 1 hour)
param delta_t default period_duration / 1000000;  # Constant to convert m3/s to Hm3 per period


### VARIABLES
var Generation {GENERATORS, PERIODS} >= 0;  # MW
var ReservoirStorage {RESERVOIR_HYDRO_GENERATORS, PERIODS} >= 0;  # Hm3
var TurbiningFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s
var Spillage {HYDRO_GENERATORS, PERIODS} >= 0;  # MWh
var SpillageFlow {HYDRO_GENERATORS, PERIODS} >= 0;  # m3/s

# Thermal Unit Commitment Variables
var IsOnline {THERMAL_GENERATORS, PERIODS} binary;  # 1 if generating, 0 otherwise
var StartUp {THERMAL_GENERATORS, PERIODS} binary;   # 1 if turned on in period t
var ShutDown {THERMAL_GENERATORS, PERIODS} binary;  # 1 if turned off in period t

# Efficiency as a function of Reservoir Storage (QUAD)
var ScaledReservoirStorage {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = ReservoirStorage[g,t]/1000; # Scale to match function
var Efficiency_QUAD {g in RESERVOIR_QUAD_EFF_GENERATORS, t in PERIODS} = 
    quad_coeff[g,1] * (ScaledReservoirStorage[g,t])^2 + quad_coeff[g,2] * ScaledReservoirStorage[g,t] + quad_coeff[g,3];

# BESS Variables
var Charge {BESS, PERIODS} >= 0;     # MW
var Discharge {BESS, PERIODS} >= 0;  # MW
var SoC {g in BESS, t in PERIODS} >= 0 <= bess_capacity[g];        # State of Charge (MWh)
var IsCharging {BESS, PERIODS} binary;

### OBJECTIVE FUNCTION
maximize TotalProfit:
    sum {g in GENERATORS, t in PERIODS} (market_price[t] * Generation[g, t])
    + sum {b in BESS, t in PERIODS} (market_price[t] * Discharge[b, t])
    - sum {b in BESS, t in PERIODS} (market_price[t] * Charge[b, t])
    - sum {g in THERMAL_GENERATORS, t in PERIODS} (marginal_cost[g, t] * Generation[g, t])
    - sum {g in HYDRO_GENERATORS, t in PERIODS} (penalty_spillage[g] * Spillage[g, t]);


### CONSTRAINTS

# --- Thermal Unit Commitment Constraints ---

subject to ThermalStateLogic {g in THERMAL_GENERATORS, t in PERIODS}:
    IsOnline[g, t] - (if t = first(PERIODS) then initial_online[g] else IsOnline[g, prev(t)]) 
    = StartUp[g, t] - ShutDown[g, t];

subject to MinUpTimeLimit {g in THERMAL_GENERATORS, t in PERIODS}:
    sum {tp in PERIODS: ord(tp) >= ord(t) - min_up_time[g] + 1 and ord(tp) <= ord(t)} 
        StartUp[g, tp] <= IsOnline[g, t];

subject to MinDownTimeLimit {g in THERMAL_GENERATORS, t in PERIODS}:
    sum {tp in PERIODS: ord(tp) >= ord(t) - min_down_time[g] + 1 and ord(tp) <= ord(t)} 
        ShutDown[g, tp] <= 1 - IsOnline[g, t];

subject to ThermalGenerationMax {g in THERMAL_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t] * IsOnline[g, t];

subject to ThermalGenerationMin {g in THERMAL_GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t] * IsOnline[g, t];

subject to ThermalRampUpLimit {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, t] - Generation[g, prev(t)] <= 
        ramp_up_rate[g] * IsOnline[g, prev(t)] + max_generation[g, t] * StartUp[g, t];

subject to ThermalRampDownLimit {g in THERMAL_GENERATORS, t in PERIODS: ord(t) > 1}:
    Generation[g, prev(t)] - Generation[g, t] <= 
        ramp_down_rate[g] * IsOnline[g, t] + max_generation[g, prev(t)] * ShutDown[g, t];

# --- Hydro Constraints ---

subject to ReservoirHydroBalance {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] = (if t = first(PERIODS) then initial_storage[g] else ReservoirStorage[g, prev(t)])
                                + delta_t * (inflows[g, t]
                                            + sum {ug in UPSTREAM_GENERATORS[g]} (TurbiningFlow[ug, t] + SpillageFlow[ug, t])
                                            - TurbiningFlow[g, t]
                                            - SpillageFlow[g, t]);

subject to RunOfRiverBalance {g in ROR_HYDRO_GENERATORS, t in PERIODS}:
    TurbiningFlow[g, t] + SpillageFlow[g, t] <= inflows[g, t];

subject to ReservoirStorageLimit {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    ReservoirStorage[g, t] <= reservoir_max_capacity[g];

subject to ReservoirFinalStorage {g in RESERVOIR_HYDRO_GENERATORS}:
    ReservoirStorage[g, last(PERIODS)] = final_storage[g];

subject to HydroGenerationFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                        then Efficiency_QUAD[g, t] 
                        else efficiency[g]) * TurbiningFlow[g, t];

subject to HydroSpillageFromFlow {g in HYDRO_GENERATORS, t in PERIODS}:
    Spillage[g, t] = (if g in RESERVOIR_QUAD_EFF_GENERATORS 
                      then Efficiency_QUAD[g, t] 
                      else efficiency[g]) * SpillageFlow[g, t];

subject to HydroGenerationMax {g in RESERVOIR_HYDRO_GENERATORS, t in PERIODS}:
    Generation[g, t] <= max_generation[g, t];

# --- General Generation Constraints ---

subject to RenewableGenerationLimit {g in RENEWABLE_GENERATORS, t in PERIODS}:
    Generation[g, t] <= renewable_resource[g, t];

subject to NonThermalGenerationMin {g in GENERATORS diff THERMAL_GENERATORS, t in PERIODS}:
    Generation[g, t] >= min_generation[g, t];

# --- BESS Constraints ---

subject to ChargeMaxLimit {g in BESS, t in PERIODS}:
    Charge[g, t] <= max_bess_power[g] * IsCharging[g, t];

subject to ChargeMinLimit {g in BESS, t in PERIODS}:
    Charge[g, t] >= min_charging_power[g] * IsCharging[g, t];

subject to DischargeLimit {g in BESS, t in PERIODS}:
    Discharge[g, t] <= max_bess_power[g] * (1 - IsCharging[g, t]);

subject to SoC_Balance {g in BESS, t in PERIODS}:
    SoC[g, t] = (if t = first(PERIODS) then initial_soc[g] else SoC[g, prev(t)])
                + (Charge[g, t] * (bess_efficiency[g]^0.5)) 
                - (Discharge[g, t] / (bess_efficiency[g]^0.5));

subject to MinChargingPeriods {g in BESS, t in PERIODS: ord(t) > 1}:
    (IsCharging[g,t] == 1 and IsCharging[g, prev(t)] == 0) ==> forall {tp in t..min(t+min_charging_periods[g]-1, last(PERIODS))} IsCharging[g, tp] == 1;

There are a couple **lessons** to draw from this challenge:
* Use tools that allow you to grow as your model does so. Real world is dynamic, complex and always changing. Using inadequate tools can be a drag further down the road.
* Don't be afraid to try out different formulations that may intuitively seem more complex or harder to solve (remember the PWL vs Quadratic for this example). AMPL allows you to try out several formulations keeping changes to a minimum and preserving model readability by using special MP operators.
* It is always best to start with a simple model and do incremental adjustments to it, as it has been done so far in the workshop.
* Keep track of the characteristics of your model as it evolves. After the modifications done in this section, the model went from being a LP to a non-convex MINLP, and proper solving tools should be used for it.

## Step 3. Project Restrictions

Any project development will have to face limiting issues in aspects like licensing budget, execution time, accuracy of the solution, etc.

A great aspect about AMPL is that it is solver agnositc, so you're free to try out any solver that may adjust to your porject specific requirements. Moreover, AMPL allows you to access any parameter the solver exposes to control and tune its behaviour to your specific context.

For our energy example, suppose the requirements state that each scenario must be solved under two minutes, given certain hardware conditions.

![Solver Benchmark results](img/solver_benchmark.png)

After running the benchmark, it is noted that adding UC constraints make the model really hard to solve. One can argue that this constraints would not have much impact in the medium to long term (scope of this model) but are more relevant in a short term horizon. This type of trade-offs are pretty common when developing real prescriptive analytics solutions and the correct way to handle them is providing options to the stakeholders or decision makers, that should be involved since the first versions of the model.

Moreover, the process of developing different versions of the model can benefit from DecisionOps platforms and CI/CD frameworks such as the one available on Nextmv, an AMPL partner with full integration capabilities.

This step leaves **lessons** like:
* Trying out several options that adapt to your model or solutuon requirements is crucial to stay within your project limits.
* When tuning a solver always compare against a baseline. Additionally, let the log guide the tuning process or use the automatic tools some solvers provide if time and compute power is available.
* Don't underestimate the impact that scaling and reformulating can have in model performance. It is time well spent.
* Your model won't be able to solve any new requirement that arises. Trade offs are part of the process and the task at hand is to be able to clearly communicate and evalute the alternatives. 

## Step 4. Gaining Trust and Adoption

Transitioning from well-established decision process to a different tool can be difficult and adoption is far from guaranteed. Besides including the decision makers in the process from the beginning, there are a couple additional tools under your belt:

1. Answering "what-if" or "why not" questions.
    * AMPL snapshots save every aspect of the model including data, options and solve status. Hence, after exporting a snapshot, one can reload it to explore additional questions that stakeholders may have.
3. Provide access to solver special features such as solution pools or feasibility analysis.
    * AMPL lets you easily access these features configuring the right options. 

Our solve function includes a particular argument `export_model`.

When set to `True`, this argument call the snapshot feature and the model is exported.

```python
if export_model:
    ampl.snapshot(f"model_snapshot_{scenario}.run")
```

To reload that model, the simple `read` command is enough.

```python
ampl.read(f"model_snapshot_{scenario}.run")
```

## Step 5. Integration and Deployment Wall

The final challenge you'll probably face when taking a model to production is related to the already in-place infrastructure.

In this sense, the company may already have systems deployed either using cloud services of self-hosted solutions.

Having the flexibility to work in a frictionless manner in any of these environments is highly relevant in today's tech landscape.

On the other hand, being integrated to robust development frameworks or flexible modeling languages provides a great advantage in deploying the solution. Using the right tools allow you to modularize your solution adn provide easy communication interfaces (APIs) to seamlessly integrate to any other system.

## Conclusion and Lessons Learned

Throughout the workshop, we've faced many of the most common challenges that arise when developing a model and taking it to a real productive decision system or process.

![Summary of workshop](img/closing.jpg)

It is worth mentioning that the list of steps or challenges treated in this workshop is not exhaustive and many others exist. One of the most prominent ones is facing the uncertainty naturally linked to the random variables involved in the decision process. Although not extensively covered in this workshop, AMPL can adjust well to classical approaches such as stochastic and dynamic programming, or newer frameworks like sequential decision analytics (SDA).